In [ ]:
def search_chains(current: Board, max_length: int = 8):
    links = set(search_hard_segms(current)) | set(search_hard_cells(current))

    counts = count_finals(current)

    def rate(link: Link):
        return max(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain.init(l) for l in links]

    print(list(map(str, init)))

    def expanding(chain: Chain):
        yield from expand_alc(chain, links)

    def matching(chain: Chain):
        return match_loop(chain) or match_rope(chain)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)


def chains(current: Board) -> Resolving:
    for chain in search_chains(current):
        if match_loop(chain):
            res = tuple(resolve_loop(current, chain))
        elif match_rope(chain):
            res = tuple(resolve_rope(current, chain))
        else:
            res = None

        if not res:
            continue  # if didn't work

        yield from res
        break  # on first worked

In [ ]:
def search_ultrachains(current: Board, max_length: int = 8):
    links = set(search_hard_segms(current)) | set(search_hard_cells(current)) | set(search_hard_groups(current))

    counts = count_finals(current)

    def rate(link: Link):
        return max(counts[link[0].dig], counts[link[1].dig])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain.init(l) for l in links]

    print(list(map(str, init)))

    def expanding(chain: Chain):
        yield from expand_alc(chain, links)

    def matching(chain: Chain):
        return match_loop(chain) or match_rope(chain)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

In [ ]:
def uchains(current: Board) -> Resolving:
    for chain in search_ultrachains(current):
        if match_loop(chain):
            res = tuple(resolve_loop(current, chain))
        elif match_rope(chain):
            res = tuple(resolve_rope(current, chain))
        else:
            res = None

        if not res:
            continue  # if didn't work

        yield from res
        break  # on first worked

In [ ]:
def scan_zonegroups(board: Board, zone: Zone, dig: int) -> Iterable[Node]:
    for side in Zone.across(zone):
        sect: Zone = zone & side  # type: ignore
        drafts = tuple(filter(lambda c: dig in c, draftborhood(board, sect)))
        if len(drafts) == 1:
            [single] = drafts
            yield Node.at(single.loc, dig)
        elif len(drafts) > 1:
            yield Node.at(sect, dig)

In [ ]:
def search_hard_groups(board: Board) -> Iterable[HLink]:
    for box in Zone.Allbox():
        drafts = set(draftborhood(board, box))
        counts = count_digits(drafts)
        for dig in counts.keys():  # including cnt = 1
            # print(box, dig, counts[dig])
            groups = set(scan_zonegroups(board, box, dig))
            occupied = set(c.loc for c in drafts if dig in c)

            # internal links: disjoint groups covering all drafts in the box
            for g1, g2 in itercomb(groups, 2):
                g1cov = set(iter(g1.zone))
                g2cov = set(iter(g2.zone))
                if not (g1cov & g2cov) and g1cov | g2cov == occupied:
                    yield HLink((g1, g2))

            # side links: to any of side units
            for g in groups:
                for side in Zone.aside(box, g.zone):
                    sidegroups = set(scan_zonegroups(board, side, dig))
                    if len(sidegroups) == 2:
                        g1, g2 = sidegroups
                        yield HLink((g1, g2))

In [ ]:
def check_gsoft(n1: Node, n2: Node):
    assert n1 != n2
    return n1.dig == n2.dig and visibility(n1.zone, n2.zone)

In [1]:
# from analysis import Group

In [2]:
# def scan_groups(board: Board, dig: int):
#     for box in Zone.Allbox():
#         for side in Zone.across(box):
#             zone: Zone = box & side  # type: ignore impossible None
#             drafts = frozenset(filter(lambda c: dig in c, draftborhood(board, zone)))
#             if len(drafts):
#                 yield Group((box, side), dig, drafts)

### locked

A group which is alone in a unit is always true.

Rule is the same as for open singles:

- remove all conflicting drafts (in all units where the group is fully visible)


In [3]:
# def resolve_locked(board: Board, group: Group):
#     # print(tuple(map(str, group.zones)), tuple(map(str, group.cells)))

#     sidebours = {
#         z: tuple(
#             filter(
#                 lambda c: group.dig in c and c not in group.cells,
#                 draftborhood(board, z),
#             )
#         )
#         for z in group.zones
#     }

#     counts = Counter({z: len(sidebours[z]) for z in group.zones})

#     # for z in group.zones:
#     #     print(z, list(map(str, sidebours[z])))

#     # the last one is where it's possibly = 0
#     ((z1, cnt1), (z0, cnt0)) = counts.most_common()

#     if cnt0 == 0 and cnt1 > 0:
#         yield Resolution(
#             castaways={Node.at(c, group.dig) for c in sidebours[z1]},
#             highlights={"empties": {Node(z0, group.dig)}, "anchors": {group.node()}},
#         )


# def locked_groups(board: Board) -> Resolving:
#     counts = count_digits(draftboard(board))

#     for dig, cnt in reversed(counts.most_common()):
#         # print(dig, cnt, "...")
#         for grp in scan_groups(board, dig):
#             # print("...", *map(str, grp.cells))
#             if len(grp.cells) == 1:
#                 continue
#             yield from resolve_locked(board, grp)

In [4]:
# for _ in locked_groups(puzzle):
#     pprint(_)